In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
from openai import OpenAI
from dotenv import load_dotenv
import warnings
import joblib

warnings.filterwarnings('ignore')

# 환경변수 로드
load_dotenv()

print("=" * 80)
print("Agent #1: 최적 CF 해석 및 정당성 설명 (Interpreter)")
print("=" * 80)

# ============================================================================
# 1. 설정 및 데이터 로드
# ============================================================================
FEATURE_MAPPING = {
    'FN1_5': '유형자산', 'FN3_10': '청산가치율', 'FN1_4': '재고자산',
    'FN2_10_1': '전기당기순이익', 'FN2_1': '매출액', 'FN2_7': '영업외수익',
    'FN1_13_1': '전기자산총계', 'FN1_11_1': '전기매출채권', 'FN3_6': '적립금비율',
    'FN2_1_1': '전기매출액', 'FN1_13': '자산총계', 'FN1_11': '매출채권',
    'FN1_19': '부채총계', 'FN1_16': '차입금', 'FN3_3': '부채상환계수',
    'PERF_12M': '부도여부'
}

# 데이터 로드
try:
    # Step 4에서 생성된 최적 CF 파일
    df_filtered = pd.read_csv('cf_results_filtered.csv')
    
    # Step 4에서 생성된 전체 평가 내역 (비교 설명을 위해 필요)
    df_details = pd.read_csv('cf_evaluation_details.csv')
    
    selected_features = joblib.load('selected_features_final.pkl')
    
    print(f"분석 대상(최적 CF): {len(df_filtered)}건")
    print(f"참조 대상(전체 후보): {len(df_details)}건")
    
except FileNotFoundError:
    raise FileNotFoundError("필요한 CSV/PKL 파일이 없습니다. 이전 Step들을 먼저 실행하세요.")

# ============================================================================
# 2. Agent 클래스 정의
# ============================================================================
class CFInterpreter:
    def __init__(self):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = os.getenv('LLM_MODEL', 'gpt-4o-mini')

    def generate_interpretation(self, best_row, all_candidates):
        """
        최적 CF에 대한 해석 리포트 생성
        """
        company_id = best_row['ID']
        
        # 1. 정량적 비교 데이터 구성
        # 다른 후보들과 점수 비교
        other_cfs = all_candidates[all_candidates['ID'] == company_id]
        avg_score = other_cfs['Quality_Score'].mean()
        best_score = best_row['Quality_Score']
        
        # 주요 변화량 추출 (Top 3)
        changes = {}
        for feat in selected_features:
            delta = best_row[f'Change_{feat}']
            if abs(delta) > 0.001:
                korean_name = FEATURE_MAPPING.get(feat, feat)
                changes[korean_name] = {
                    'original': best_row[f'Original_{feat}'],
                    'target': best_row[f'CF_{feat}'],
                    'delta': delta
                }
        
        # 변화량이 큰 순서로 정렬
        sorted_changes = dict(sorted(changes.items(), key=lambda item: abs(item[1]['delta']), reverse=True)[:5])

        # 2. 프롬프트 작성
        system_prompt = """
        당신은 기업신용평가 전문가입니다. 
        AI 모델이 제안한 '재무구조 개선 시나리오(Counterfactual)'가 왜 선택되었는지, 그리고 그 의미가 무엇인지 설명해야 합니다.
        """
        
        user_prompt = f"""
        [기업 ID: {company_id}]에 대한 AI 재무 개선 제안서입니다.
        
        # 1. 선정 근거 (Why this scenario?)
        이 시나리오는 총 {len(other_cfs)}개의 후보 중 가장 높은 품질 점수({best_score:.2f}점, 평균 {avg_score:.2f}점)를 기록하여 선정되었습니다.
        - 현실성(Realism): {best_row['Realism']:.2f} (유사 기업 분포와 일치하는 정도)
        - 변화 최소화(Proximity): {best_row['Proximity']:.4f} (적은 변화로 큰 효과 달성)
        - 강건성(Robustness): {best_row['Robustness']:.2f} (시장 변동에도 승인 유지 확률)
        
        # 2. 핵심 제안 내용 (What to change?)
        다음 변수들의 조정이 필요합니다:
        {json.dumps(sorted_changes, ensure_ascii=False, indent=2)}
        
        # 요청사항
        위 데이터를 바탕으로 다음 JSON 형식의 해설 리포트를 작성하세요.
        
        {{
            "selection_reason": "이 시나리오가 선정된 이유를 품질 지표(현실성, 효율성 등)를 들어 1문장으로 요약",
            "business_meaning": "제안된 수치 변화가 실제 경영상 어떤 액션(예: 자산 매각, 차입금 상환 등)을 의미하는지 설명 (2~3문장)",
            "feasibility_assessment": "이 변화가 도매업 특성상 현실적으로 실현 가능한 수준인지 평가 (Pass/Warning)",
            "expected_outcome": "이대로 이행 시 예상되는 긍정적 효과 (부도 확률 감소 등)"
        }}
        """

        # 3. API 호출
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.3 
            )
            return json.loads(response.choices[0].message.content)
            
        except Exception as e:
            return {"error": str(e)}

# ============================================================================
# 3. 실행 및 저장
# ============================================================================
if __name__ == "__main__":
    agent = CFInterpreter()
    results = []
    
    print(f"\n총 {len(df_filtered)}개 기업에 대한 해석 생성 시작...")
    
    # 시간 관계상 샘플 5개만 테스트하려면 아래 슬라이싱 사용 (전체 실행 시 제거)
    target_rows = df_filtered.iterrows() # .iloc[:5].iterrows() 
#    target_rows = df_filtered.iloc[:5].iterrows()
    
    from tqdm import tqdm
    for idx, row in tqdm(target_rows, total=len(df_filtered)):
        
        interpretation = agent.generate_interpretation(row, df_details)
        
        # 결과 합치기
        result_entry = row.to_dict()
        result_entry.update(interpretation) # LLM 응답 추가
        results.append(result_entry)
        
    # DataFrame 변환
    final_df = pd.DataFrame(results)
    
    # 결과 저장
    output_filename = 'agent1_interpretation_results.csv'
    final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print("\n" + "=" * 80)
    print("해석 생성 완료")
    print("=" * 80)
    print(f"저장 파일: {output_filename}")
    
    # 샘플 출력
    if not final_df.empty:
        sample = final_df.iloc[0]
        print(f"\n[ID: {sample['ID']} 해석 결과]")
        print(f"1. 선정 이유: {sample.get('selection_reason')}")
        print(f"2. 비즈니스 의미: {sample.get('business_meaning')}")
        print(f"3. 현실성 평가: {sample.get('feasibility_assessment')}")

Agent #1: 최적 CF 해석 및 정당성 설명 (Interpreter)
분석 대상(최적 CF): 266건
참조 대상(전체 후보): 1061건

총 266개 기업에 대한 해석 생성 시작...


100%|████████████████████████████████████████████████████████████████████████████████| 266/266 [18:45<00:00,  4.23s/it]


해석 생성 완료
저장 파일: agent1_interpretation_results.csv

[ID: 75.0 해석 결과]
1. 선정 이유: 이 시나리오는 높은 품질 점수(0.91점)를 기록하며, 현실성과 변화 최소화 측면에서 유사 기업과의 일치도가 높고, 시장 변동성에도 강건성을 유지할 수 있어 선정되었습니다.
2. 비즈니스 의미: 제안된 수치 변화는 매출액 감소와 함께 자산의 매각 및 차입금 상환을 포함합니다. 이는 기업이 재무구조를 개선하고, 부채 부담을 줄이며, 자산 효율성을 높이기 위한 전략적 조치를 의미합니다.
3. 현실성 평가: 이 변화는 도매업의 특성상 자산 매각 및 차입금 상환이 가능하므로 현실적으로 실현 가능한 수준으로 평가됩니다. (Pass)
